# Model Experiment — `ARIMA`

## Theory

**ARIMA(p, d, q)** models a single series as a linear function of its own past
values (AutoRegressive, order `p`), its own past forecast errors (Moving Average,
order `q`), after differencing `d` times to make it stationary (Integrated).
It has no built-in concept of "other stores", "other departments", holidays, or
promotions — it only sees one number per week.

**Why here:** every (Store, Dept) pair is its own weekly time series, so the
natural way to apply ARIMA is one independent model *per series* (~3,000 series).
That is fundamentally different from the tabular, all-series-at-once approach
used for the tree/DL models — there is no shared `feature_selection`-style tabular
matrix here; "feature selection" for ARIMA really means deciding the
differencing order and whether the series needs external (exogenous) regressors
at all (see Run 2).

**Expected limitations:** ARIMA (no seasonal term) cannot represent the strong
yearly seasonality (Thanksgiving/Christmas spikes) found in the EDA — that's what
the SARIMA notebook adds. It also can't use Store `Type`/`Size`, markdowns, CPI,
etc., and it can't share information across series (a short/noisy series gets a
bad fit with nothing to borrow strength from). It should mainly serve as an
interpretable statistical baseline for comparison against the global tree/DL models.

**Per the assignment brief:** ARIMA/SARIMA are older, well-understood models —
more weight goes on the theoretical discussion than on exhaustive tuning, so the
hyperparameter search below is intentionally small and guided by ACF/PACF
diagnostics rather than a brute-force grid.

**Tracking:** MLflow (this is a classical statistical model, not a neural net, so
it follows the same MLflow experiment/run structure as the tree-based models —
Wandb is only offered as an alternative for the DL architectures).

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    RAW_DIR = KAGGLE_INPUT / KAGGLE_COMPETITION
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Per-series ARIMA/SARIMA Pipeline building blocks

In [ ]:
import warnings
from statsmodels.tsa.arima.model import ARIMA
from sklearn.base import BaseEstimator, RegressorMixin

FREQ = 'W-FRI'

def build_series_panel(df, value_col='Weekly_Sales'):
    """{(Store,Dept): continuous weekly pd.Series}, small gaps linearly interpolated."""
    panel = {}
    for key, g in df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').drop_duplicates('Date')
        s = g.set_index('Date')[value_col]
        full_idx = pd.date_range(s.index.min(), s.index.max(), freq=FREQ)
        s = s.reindex(full_idx)
        s = s.interpolate(limit=4).ffill().bfill()
        panel[key] = s
    return panel

def build_exog_panel(df, exog_cols):
    """{(Store,Dept): continuous weekly pd.DataFrame of exog cols}, aligned to same index."""
    if not exog_cols:
        return {}
    panel = {}
    for key, g in df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').drop_duplicates('Date')
        e = g.set_index('Date')[exog_cols].astype(float)
        full_idx = pd.date_range(e.index.min(), e.index.max(), freq=FREQ)
        e = e.reindex(full_idx).ffill().bfill()
        panel[key] = e
    return panel


class PerSeriesForecaster(BaseEstimator, RegressorMixin):
    """Fits one statsmodels ARIMA/SARIMAX per (Store,Dept) series and predicts by
    aligning requested Dates to forecast steps ahead of that series' training window.
    Falls back to a recent-mean / Dept-mean / global-mean predictor for series that
    are new (cold-start in test), too short, or fail to converge."""

    def __init__(self, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), exog_cols=None, min_obs=20):
        self.order = order
        self.seasonal_order = seasonal_order
        self.exog_cols = exog_cols
        self.min_obs = min_obs

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)
        panel = build_series_panel(df, 'Weekly_Sales')
        exog_panel = build_exog_panel(df, self.exog_cols) if self.exog_cols else {}

        self.models_ = {}
        self.last_date_ = {}
        self.fallback_value_ = {}
        n_failed = 0
        for key, s in panel.items():
            self.fallback_value_[key] = float(s.tail(8).mean()) if len(s) else 0.0
            if len(s) < self.min_obs:
                self.models_[key] = None
                self.last_date_[key] = s.index.max()
                continue
            exog = exog_panel.get(key)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    model = ARIMA(
                        s.values, order=self.order, seasonal_order=self.seasonal_order,
                        exog=(exog.values if exog is not None else None),
                        enforce_stationarity=False, enforce_invertibility=False,
                    )
                    res = model.fit()
                self.models_[key] = res
            except Exception:
                self.models_[key] = None
                n_failed += 1
            self.last_date_[key] = s.index.max()

        self.dept_mean_ = df.groupby('Dept')['Weekly_Sales'].mean()
        self.global_mean_ = float(df['Weekly_Sales'].mean())
        self.n_series_ = len(panel)
        self.n_failed_ = n_failed
        return self

    def _fallback(self, key, store, dept):
        if key in self.fallback_value_:
            return self.fallback_value_[key]
        if dept in self.dept_mean_.index:
            return float(self.dept_mean_.loc[dept])
        return self.global_mean_

    def predict(self, X):
        preds = np.empty(len(X), dtype=float)
        X = X.reset_index(drop=True)
        for (store, dept), idx in X.groupby(['Store', 'Dept']).groups.items():
            key = (store, dept)
            rows = X.loc[idx]
            if key not in self.models_ or self.models_[key] is None:
                preds[idx] = self._fallback(key, store, dept)
                continue
            res = self.models_[key]
            last_date = self.last_date_[key]
            dates = pd.to_datetime(rows['Date'])
            steps = ((dates - last_date).dt.days // 7).to_numpy()
            max_step = int(steps.max()) if len(steps) else 0
            if max_step < 1:
                preds[idx] = self._fallback(key, store, dept)
                continue
            future_exog = None
            if self.exog_cols:
                future_exog = np.zeros((max_step, len(self.exog_cols)))
                for s, dt in zip(steps, dates):
                    if 1 <= s <= max_step:
                        match = rows.loc[rows['Date'] == dt, self.exog_cols].astype(float)
                        if len(match):
                            future_exog[s - 1] = match.values[0]
            try:
                fc_mean = res.get_forecast(steps=max_step, exog=future_exog).predicted_mean
            except Exception:
                fc_mean = np.full(max_step, self._fallback(key, store, dept))
            for pos, s in zip(idx, steps):
                if s < 1:
                    preds[pos] = self._fallback(key, store, dept)
                else:
                    preds[pos] = fc_mean[min(int(s), max_step) - 1]
        return preds

## 4. MLflow tracking setup

In [ ]:
import mlflow

def init_tracking(experiment=None):
    uri = MLFLOW_TRACKING_URI
    if uri:
        mlflow.set_tracking_uri(uri)
        if MLFLOW_TRACKING_USERNAME:
            os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
        if MLFLOW_TRACKING_PASSWORD:
            os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD
    else:
        mlflow.set_tracking_uri((WORKING_DIR / 'mlruns').as_uri())
    if experiment:
        mlflow.set_experiment(experiment)
    return mlflow.get_tracking_uri()

import numpy as np, pandas as pd
MODEL_NAME = 'ARIMA'
EXPERIMENT = f'{MODEL_NAME}_Training'
uri = init_tracking(EXPERIMENT)
print('MLflow ->', uri, '| experiment:', EXPERIMENT)

## Load raw data

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning
Per-series diagnostics: how many (Store, Dept) series exist, how many have gaps
in their weekly index, and the usual negative-sales note.

In [ ]:
n_series = raw_train.groupby(['Store', 'Dept']).ngroups
lengths = raw_train.groupby(['Store', 'Dept']).size()
span_weeks = raw_train.groupby(['Store', 'Dept'])['Date'].agg(lambda d: (d.max() - d.min()).days // 7 + 1)
n_with_gaps = int((lengths < span_weeks).sum())

with mlflow.start_run(run_name=f'{MODEL_NAME}_Cleaning'):
    mlflow.log_param('n_series', n_series)
    mlflow.log_param('n_series_with_gaps', n_with_gaps)
    mlflow.log_param('gap_fill_strategy', 'reindex_to_weekly+interpolate(limit=4)+ffill/bfill')
    mlflow.log_param('cold_start_fallback', 'recent_mean -> dept_mean -> global_mean')
    mlflow.log_metric('n_train_rows', len(raw_train))
    mlflow.log_metric('n_negative_sales', int((raw_train.Weekly_Sales < 0).sum()))
    print(f'{n_series} series, {n_with_gaps} with missing weeks (interpolated)')

## Run 2 — "Feature Selection" (order & stationarity diagnostics)
For a per-series classical model, "feature selection" is: decide the
differencing order `d` via stationarity tests, and read `p`/`q` off the
ACF/PACF plots of a representative aggregate series. Plain ARIMA has no
seasonal or exogenous terms — that's the SARIMA notebook's job.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

agg = raw_train.groupby('Date')['Weekly_Sales'].sum()

adf_level = adfuller(agg)[1]
adf_diff1 = adfuller(agg.diff().dropna())[1]
print(f'ADF p-value, level: {adf_level:.4f} | after 1st difference: {adf_diff1:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(agg.diff().dropna(), lags=20, ax=axes[0], title='ACF (1st diff, aggregate)')
plot_pacf(agg.diff().dropna(), lags=20, ax=axes[1], title='PACF (1st diff, aggregate)')
plt.tight_layout(); plt.show()

candidate_orders = [(1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 1)]
exog_cols = None

with mlflow.start_run(run_name=f'{MODEL_NAME}_Feature_Selection'):
    mlflow.log_param('adf_pvalue_level', adf_level)
    mlflow.log_param('adf_pvalue_diff1', adf_diff1)
    mlflow.log_param('chosen_d', 1)
    mlflow.log_param('candidate_orders', candidate_orders)
    mlflow.log_param('exog_cols', exog_cols)
print('candidate orders:', candidate_orders)

## Run 3 — Cross-validation (small guided grid)
Per the brief's guidance not to over-invest training time in ARIMA/SARIMA, the
order search runs on a random **subsample of series** (not all ~3,000) using a
single holdout split. Widen `SAMPLE_SIZE` if you have more time/compute.

In [ ]:
SAMPLE_SIZE = 60
rng = np.random.RandomState(RANDOM_SEED)
all_keys = list(raw_train.groupby(['Store', 'Dept']).groups.keys())
sample_keys = [all_keys[i] for i in rng.choice(len(all_keys), size=min(SAMPLE_SIZE, len(all_keys)), replace=False)]
sample_mask = raw_train.set_index(['Store', 'Dept']).index.isin(sample_keys)
sample_df = raw_train[sample_mask].reset_index(drop=True)

tr_idx, va_idx = time_holdout(sample_df, weeks=8)
tr, va = sample_df.iloc[tr_idx], sample_df.iloc[va_idx]

grid_scores = {}
with mlflow.start_run(run_name=f'{MODEL_NAME}_CV'):
    mlflow.log_param('sample_size_series', len(sample_keys))
    for order in candidate_orders:
        model = PerSeriesForecaster(order=order, seasonal_order=(0, 0, 0, 0), exog_cols=exog_cols)
        model.fit(tr, tr['Weekly_Sales'])
        pred = model.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        grid_scores[order] = score
        mlflow.log_metric(f'wmae_order_{order}', score)
        print(f'order={order}: holdout WMAE={score:,.1f} (n_failed={model.n_failed_}/{model.n_series_})')
    best_order = min(grid_scores, key=grid_scores.get)
    mlflow.log_param('best_order', best_order)
    mlflow.log_metric('best_wmae_sample', grid_scores[best_order])
print('best order:', best_order, '| WMAE:', grid_scores[best_order])

## Run 4 — Final fit + log Pipeline
Refit the chosen order on **all** series (raw_train) and log the Pipeline —
this step fits ~3,000 independent ARIMA models, so it will take a while;
that cost is expected and is exactly why the search above stayed small.

In [ ]:
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline

def make_pipeline():
    return Pipeline([('model', PerSeriesForecaster(order=best_order, seasonal_order=(0, 0, 0, 0), exog_cols=exog_cols))])

with mlflow.start_run(run_name=f'{MODEL_NAME}_Final') as run:
    mlflow.log_param('order', best_order)
    holdout_tr, holdout_va = time_holdout(raw_train, weeks=8)
    p = make_pipeline()
    p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
    hv = raw_train.iloc[holdout_va]
    holdout_pred = p.predict(hv)
    holdout_wmae = wmae(hv['Weekly_Sales'], holdout_pred, hv['IsHoliday'])
    mlflow.log_metric('holdout_wmae', holdout_wmae)

    final_pipe = make_pipeline()
    final_pipe.fit(raw_train, raw_train['Weekly_Sales'])
    example = raw_test.head(5)
    sig = infer_signature(example, final_pipe.predict(example))
    mlflow.sklearn.log_model(final_pipe, artifact_path='pipeline', signature=sig, input_example=example)
    print('holdout WMAE:', holdout_wmae, '| run_id:', run.info.run_id)

> **Registering the best model:** once you know this architecture is the overall best, register it from the Final run — either via the MLflow UI or:
> ```python
> mlflow.register_model(f'runs:/{run.info.run_id}/pipeline', 'walmart_best_model')
> ```